In [4]:
import wandb
wandb.login()

True

# Load dependencies and configurations

In [5]:
import os, random, logging
import json
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import ViTForImageClassification, ViTImageProcessor
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


In [6]:
# Define paths
folder_path = "/content/drive/MyDrive/NEU/DS5500-Data Capstone/DS5500_AIGI-Detection/AIGI-data/"
drive_zip_path = folder_path + "train_data.zip"
local_zip_path = "/content/train_data.zip"
local_extract_path = "/content/train_data"

DATA_ROOT = Path('/content/train_data')
CSV_PATH  = folder_path + 'train.csv'

In [17]:
@dataclass
class Config:
    # ── Data ──────────────────────────────────────────
    data_root   : str   = '/content/train_data'
    csv_path    : str   = folder_path + 'train.csv'
    train_size  : int   = 5_000      # None = Use all images
    val_ratio   : float = 0.20
    test_ratio  : float = 0.20
    num_workers : int   = 2

    split_json_path  : str = '/content/drive/MyDrive/vit_checkpoints/split_5k_seed42.json'

    # ── Model ─────────────────────────────────────────
    model_name             : str  = 'vit_b_16'
    freeze_backbone        : bool = True
    unfreeze_last_n_blocks : int  = 0

    # ── Training ──────────────────────────────────────
    epochs          : int   = 5 # 20
    batch_size      : int   = 64
    lr              : float = 3e-4
    backbone_lr     : float = 1e-5
    weight_decay    : float = 1e-2
    label_smoothing : float = 0.1
    patience        : int   = 5
    grad_clip       : float = 1.0
    use_amp         : bool  = True

    # ── W&B ───────────────────────────────────────────
    project  : str  = 'vit-ai-detection'
    run_name : str  = 'vit-b16-10k-linear-probe'
    tags     : list = field(default_factory=lambda: ['linear-probe', 'vit'])

    # ── Misc ──────────────────────────────────────────
    seed     : int = 42
    save_dir : str = '/content/drive/MyDrive/vit_checkpoints'

cfg = Config()
print(asdict(cfg))

# Create save directory if it doesn't exist
os.makedirs(cfg.save_dir, exist_ok=True)

{'data_root': '/content/train_data', 'csv_path': '/content/drive/MyDrive/NEU/DS5500-Data Capstone/DS5500_AIGI-Detection/AIGI-data/train.csv', 'train_size': 5000, 'val_ratio': 0.2, 'test_ratio': 0.2, 'num_workers': 2, 'split_json_path': '/content/drive/MyDrive/vit_checkpoints/split_5k_seed42.json', 'model_name': 'vit_b_16', 'freeze_backbone': True, 'unfreeze_last_n_blocks': 0, 'epochs': 5, 'batch_size': 64, 'lr': 0.0003, 'backbone_lr': 1e-05, 'weight_decay': 0.01, 'label_smoothing': 0.1, 'patience': 5, 'grad_clip': 1.0, 'use_amp': True, 'project': 'vit-ai-detection', 'run_name': 'vit-b16-10k-linear-probe', 'tags': ['linear-probe', 'vit'], 'seed': 42, 'save_dir': '/content/drive/MyDrive/vit_checkpoints'}


# Helper functions

In [8]:
import zipfile
def unzip(zip_path, extract_to):
  """Unzip folders"""
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

In [9]:
def get_symdiff(set1, set2):
  """Get the symmetric difference of 2 sets"""
  return set1.symmetric_difference(set2)

# Load raw data


*   Just use train_data. Because test_data_v2 doesn't have labels.
*   Load train_data.zip to /content/ each time for fast access.



In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os
import shutil

# 1. Copy zip from Drive to local (fast access)
if os.path.exists(drive_zip_path):
    print(f"Copying '{drive_zip_path}' to '{local_zip_path}'...")
    shutil.copy(drive_zip_path, local_zip_path)
    print("Copy completed.")
else:
    print(f"Error: File not found at {drive_zip_path}")

# 2. Unzip locally
if os.path.exists(local_zip_path):
    print(f"Unzipping to '{local_extract_path}'...")
    # Ensure the destination directory exists
    os.makedirs(local_extract_path, exist_ok=True)
    unzip(local_zip_path, local_extract_path)
    print("Unzip completed.")

    # Check content
    print(f"Content of {local_extract_path}:", os.listdir(local_extract_path))

    # Set the new root for image loading
    # Since your csv contains paths like 'train_data/xxx.jpg', we point to the parent of 'train_data'
    image_root_path = local_extract_path
    print(f"\nSet 'image_root_path' to: {image_root_path}")
else:
    print("Zip file copy failed, cannot unzip.")

Output hidden; open in https://colab.research.google.com to view.

In [14]:
df = pd.read_csv(CSV_PATH)
print(f'Total samples : {len(df)}')
print(f'Columns       : {df.columns.tolist()}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nClass balance: {df["label"].mean():.2%} AI / {1-df["label"].mean():.2%} Human')
df.head()

Total samples : 79950
Columns       : ['Unnamed: 0', 'file_name', 'label']

Label distribution:
label
1    39975
0    39975
Name: count, dtype: int64

Class balance: 50.00% AI / 50.00% Human


,Unnamed: 0,file_name,label
0,0,train_data/a6dcb93f596a43249135678dfcfc17ea.jpg,1
1,1,train_data/041be3153810433ab146bc97d5af505c.jpg,0
2,2,train_data/615df26ce9494e5db2f70e57ce7a3a4f.jpg,1
3,3,train_data/8542fe161d9147be8e835e50c0de39cd.jpg,0
4,4,train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg,1


# Load data into Dataset

We do not have labels in test set -> do not use test_data_v2 at all!

## Prepare Data Splits

Filter the dataframe based on `cfg.train_size` and split it into training, validation, and test sets according to `cfg.val_ratio` and `cfg.test_ratio` for evaluation.


In [18]:
if cfg.train_size is not None:
    df_sampled, _ = train_test_split(
    df,
    train_size=cfg.train_size,
    stratify=df["label"],
    random_state=cfg.seed)
else:
    df_sampled = df.copy()

# Calculate the sum of validation and test ratios for the first split
val_test_ratio_sum = cfg.val_ratio + cfg.test_ratio

# Load split from JSON if it exists, otherwise perform the split
if os.path.exists(cfg.split_json_path):
    with open(cfg.split_json_path, 'r') as f:
        split = json.load(f)

    df_train = df_sampled[df_sampled["file_name"].isin(split["train"])].copy().reset_index(drop=True)
    df_val   = df_sampled[df_sampled["file_name"].isin(split["val"])].copy().reset_index(drop=True)
    df_test  = df_sampled[df_sampled["file_name"].isin(split["test"])].copy().reset_index(drop=True)
    print("[Split] Loaded train/val/test split from JSON.")

# Calculate the test_size for the second split (df_temp into val and test)
# This is the proportion of the temporary set that should go to the test set
else:
    print("[Split] No existing split JSON found. Performing new train/val/test split...")
     # Split df_sampled into training and a temporary set (for validation and test)
    df_train, df_temp = train_test_split(
        df_sampled,
        test_size=val_test_ratio_sum, # proportion of data for val+test
        stratify=df_sampled['label'],
        random_state=cfg.seed
    )
    test_split_ratio_from_temp = cfg.test_ratio / val_test_ratio_sum

    # Split df_temp into validation and test sets
    df_val, df_test = train_test_split(
        df_temp,
        test_size=test_split_ratio_from_temp, # proportion of df_temp for test
        stratify=df_temp['label'],
        random_state=cfg.seed
    )

    # Save the split to a JSON file
    split = {
        "train": df_train["file_name"].tolist(),
        "val": df_val["file_name"].tolist(),
        "test": df_test["file_name"].tolist()
    }

    with open(cfg.split_json_path, 'w') as f:
        json.dump({"seed": cfg.seed, "val_ratio": cfg.val_ratio, "test_ratio": cfg.test_ratio, **split}, f, indent=2)

print(f"Train set size: {len(df_train)}")
print(f"Validation set size: {len(df_val)}")
print(f"Test set size: {len(df_test)}")

[Split] No existing split JSON found. Performing new train/val/test split...
Train set size: 3000
Validation set size: 1000
Test set size: 1000


## Define Image Transformations

Create `torchvision.transforms` for data augmentation (like random horizontal flip, color jitter) and preprocessing (resize, center crop, normalize) suitable for the model, applying different transformations for training and validation/test sets. We do this to make the task harder in training which forces the model to learn more robust features.


In [19]:
import torchvision.transforms as transforms

# ImageNet mean and std (common for pre-trained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transformations with data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224), # Randomly crop and resize to 224x224
    transforms.RandomHorizontalFlip(), # Randomly flip the image horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # Randomly change brightness, contrast, saturation, and hue
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) # Normalize with ImageNet mean and std
])

# Validation and Test transformations (preprocessing only, no augmentation)
val_test_transforms = transforms.Compose([
    transforms.Resize(256), # Resize the image to 256x256
    transforms.CenterCrop(224), # Crop the center of the image to 224x224
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) # Normalize with ImageNet mean and std
])

print("Defined train_transforms and val_test_transforms.")

Defined train_transforms and val_test_transforms.


## Implement Custom Dataset

Create a `torch.utils.data.Dataset` class that handles loading images from the specified `cfg.data_root` using `PIL.Image`, applies the defined transformations, and returns the processed image and its corresponding label.


In [20]:
class AIDataset(Dataset):
    def __init__(self, dataframe, data_root, transform=None):
        self.dataframe = dataframe
        self.data_root = data_root
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['file_name']
        label = self.dataframe.iloc[idx]['label']

        # Construct full image path. Split 'train_data/' from the filename if present.
        # This ensures compatibility with both direct filenames and 'train_data/filename.jpg' paths
        if img_name.startswith('train_data/') or img_name.startswith('test_data_v2/'):
            img_name = os.path.basename(img_name)

        img_path = os.path.join(self.data_root, img_name)

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

# Instantiate datasets
train_dataset = AIDataset(dataframe=df_train, data_root=image_root_path, transform=train_transforms)
val_dataset = AIDataset(dataframe=df_val, data_root=image_root_path, transform=val_test_transforms)
test_dataset = AIDataset(dataframe=df_test, data_root=image_root_path, transform=val_test_transforms)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 3000
Validation dataset size: 1000
Test dataset size: 1000


## Initialize DataLoaders

### Subtask:
Create `torch.utils.data.DataLoader` instances for the training, validation, and test sets using `cfg.batch_size` and `cfg.num_workers` to efficiently load and batch the data for model training.


In [21]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True # Optimize data transfer to GPU
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

print(f"Number of batches in train_dataloader: {len(train_dataloader)}")
print(f"Number of batches in val_dataloader:   {len(val_dataloader)}")
print(f"Number of batches in test_dataloader:  {len(test_dataloader)}")

Number of batches in train_dataloader: 47
Number of batches in val_dataloader:   16
Number of batches in test_dataloader:  16


## Load and Modify ViT Model

### Subtask:
Load a pre-trained ViT model from `torchvision.models` (as `cfg.model_name` is 'vit_b_16'). Modify its final classification head to output 2 classes for binary classification. Implement conditional logic to freeze earlier layers if `cfg.freeze_backbone` is true and unfreeze the last `cfg.unfreeze_last_n_blocks` of the ViT backbone.


In [22]:
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights

# Load a pre-trained ViT-B_16 model
weights = ViT_B_16_Weights.IMAGENET1K_V1
model = models.vit_b_16(weights=weights)

# Replace classification head for 2 classes
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, 2)

# Implement conditional freezing logic
if cfg.freeze_backbone:
    print("Freezing backbone layers...")
    for param in model.parameters():
        param.requires_grad = False

    # unfreeze head
    for param in model.heads.parameters():
        param.requires_grad = True

    # unfreeze last N transformer blocks
    if cfg.unfreeze_last_n_blocks > 0:
        print(f"Unfreezing last {cfg.unfreeze_last_n_blocks} encoder blocks...")
        for blk in model.encoder.layers[-cfg.unfreeze_last_n_blocks:]:
            for param in blk.parameters():
                param.requires_grad = True

# Move the modified model to the device
model = model.to(device)

assert all(
    (not p.requires_grad)
    for name, p in model.named_parameters()
    if "heads" not in name
), "Backbone has trainable params! Freeze failed."

print("Backbone successfully frozen. Only head is trainable.")

# Print the number of trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / Total params: {total:,} ({trainable/total:.4%})")

# Print the model architecture
print("\nModel architecture after modifications:")
print(model)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 155MB/s]


Freezing backbone layers...
Backbone successfully frozen. Only head is trainable.
Trainable params: 1,538 / Total params: 85,800,194 (0.0018%)

Model architecture after modifications:
VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
   

In [23]:
import torch.nn.functional as F

# Define loss function with label smoothing
# For binary classification (2 classes), CrossEntropyLoss is appropriate.
# If labels are 0 and 1, we can pass them directly. If the output of the model
# is logits, CrossEntropyLoss handles softmax internally.

def cross_entropy_loss_with_label_smoothing(outputs, targets, num_classes=2, label_smoothing=0.1):
    # Apply label smoothing
    # For binary classification, targets are 0 or 1.
    # We need to convert them to one-hot for label smoothing if using NLLLoss with log_softmax
    # Or directly adjust probabilities for CrossEntropyLoss which expects class indices as targets

    # Convert targets to one-hot encoding for label smoothing calculation
    # CrossEntropyLoss expects class indices, so this step is just for smoothing logic.
    # For actual CE loss, we pass original targets.
    if targets.dim() == 1:
        # Convert scalar labels to one-hot for smoothing calculation
        one_hot = F.one_hot(targets.long(), num_classes=num_classes).float()
    else:
        one_hot = targets.float()

    # Smooth labels
    smooth_labels = one_hot * (1.0 - label_smoothing) + label_smoothing / num_classes

    # CrossEntropyLoss expects raw logits and integer class labels (0 or 1 for binary)
    # If targets are float (e.g., from one-hot), it expects probabilities. Here they are 0/1.
    # So, we convert original targets to long to be used by F.cross_entropy directly for true labels.
    loss = F.cross_entropy(outputs, targets.long(), label_smoothing=label_smoothing)

    return loss

# Use the custom function or directly specify label_smoothing in F.cross_entropy
criterion = lambda outputs, targets: cross_entropy_loss_with_label_smoothing(
    outputs, targets, num_classes=2, label_smoothing=cfg.label_smoothing
)

print(f"Loss function defined with label smoothing: {cfg.label_smoothing}")

Loss function defined with label smoothing: 0.1


In [24]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Separate parameters for different learning rates
# Parameters that require gradients will be trained
# Only the head and potentially unfreezed backbone layers will have requires_grad=True
optimizer_params = [
    {'params': model.heads.parameters(), 'lr': cfg.lr} # Learning rate for the classification head
]

# Add backbone parameters if they require gradients and were not frozen
# We check if any parameter in the backbone layers requires grad
backbone_params = []
for name, param in model.named_parameters():
    if param.requires_grad and 'heads' not in name: # Exclude heads layer as it's already added
        backbone_params.append(param)

if backbone_params:
    optimizer_params.append({'params': backbone_params, 'lr': cfg.backbone_lr})


# Initialize the optimizer
optimizer = optim.AdamW(optimizer_params, weight_decay=cfg.weight_decay)

# Initialize the learning rate scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)

print(f"Optimizer defined with learning rate for head: {cfg.lr}, and backbone: {cfg.backbone_lr} (if active).")
print(f"Scheduler defined with T_max: {cfg.epochs}.")

Optimizer defined with learning rate for head: 0.0003, and backbone: 1e-05 (if active).
Scheduler defined with T_max: 5.


## Implement Training and Validation Functions

### Subtask:
Define Python functions for performing one epoch of training (including forward pass, loss calculation, backward pass, optimizer step, gradient clipping with `cfg.grad_clip`, and mixed precision training with `torch.cuda.amp.autocast` and `GradScaler` if `cfg.use_amp` is true) and one epoch of validation.


In [25]:
import torch.nn.functional as F

def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler, grad_clip, use_amp):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast(enabled=use_amp):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        if use_amp:
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer) # Unscale gradients before clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device, use_amp):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            with autocast(enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            total_loss += loss.item()

            # Convert outputs to probabilities and predictions
            probs = F.softmax(outputs, dim=1)[:, 1] # Probability of class 1 (AI)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    return avg_loss, np.array(all_preds), np.array(all_labels)

print("Defined train_one_epoch and validate_one_epoch functions.")

Defined train_one_epoch and validate_one_epoch functions.


## Execute Model Training

### Subtask:
Run the main training loop for `cfg.epochs`, iterating through training and validation functions. Implement logic for tracking metrics, saving the best performing model (based on validation loss or accuracy) to `cfg.save_dir`, and applying early stopping if validation performance does not improve for `cfg.patience` epochs.


In [26]:
import wandb
import os

# 1. Initialize wandb
wandb.init(
    project=cfg.project,
    name=cfg.run_name,
    tags=cfg.tags,
    config=asdict(cfg)
)


# 2. Create an instance of torch.cuda.amp.GradScaler
scaler = GradScaler() if cfg.use_amp else None

# 3. Initialize variables for tracking best model and early stopping
best_val_loss = float('inf')
patience_counter = 0

print("Starting training...")
for epoch in range(1, cfg.epochs + 1):
    print(f"Epoch {epoch}/{cfg.epochs}")

    # 5. Training step
    train_loss = train_one_epoch(
        model, train_dataloader, criterion, optimizer, device, scaler, cfg.grad_clip, cfg.use_amp
    )

    # 6. Validation step
    val_loss, val_preds, val_labels = validate_one_epoch(
        model, val_dataloader, criterion, device, cfg.use_amp
    )

    # 7. Calculate metrics for the validation set
    val_roc_auc = roc_auc_score(val_labels, val_preds)

    # Binarize predictions for classification report and confusion matrix
    val_bin_preds = (val_preds > 0.5).astype(int)
    val_report = classification_report(val_labels, val_bin_preds, output_dict=True, zero_division=0)
    val_confusion_matrix = confusion_matrix(val_labels, val_bin_preds)

    # Extract key metrics from classification report
    val_accuracy = val_report['accuracy']
    val_precision = val_report['macro avg']['precision']
    val_recall = val_report['macro avg']['recall']
    val_f1 = val_report['macro avg']['f1-score']

    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val ROC AUC: {val_roc_auc:.4f}")
    print(f"  Val Accuracy: {val_accuracy:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val Recall: {val_recall:.4f}")
    print(f"  Val F1-Score: {val_f1:.4f}")
    print(f"  Confusion Matrix:\n{val_confusion_matrix}")

    # 8. Log metrics to wandb
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_roc_auc": val_roc_auc,
        "val_accuracy": val_accuracy,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1_score": val_f1,
        "val_confusion_matrix": wandb.Table(data=val_confusion_matrix.tolist(), columns=["Pred 0", "Pred 1"])
    })

    # 9. Step the learning rate scheduler
    scheduler.step()

    # 10. Implement logic to save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        model_save_path = os.path.join(cfg.save_dir, 'best_model.pth')
        torch.save(model.state_dict(), model_save_path)
        print(f"  Saved best model to {model_save_path} (Val Loss: {best_val_loss:.4f})")
        wandb.run.summary["best_val_loss"] = best_val_loss
        wandb.run.summary["best_val_roc_auc"] = val_roc_auc
        wandb.run.summary["best_val_accuracy"] = val_accuracy
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve. Patience: {patience_counter}/{cfg.patience}")

    # 11. Implement early stopping
    if patience_counter > cfg.patience:
        print(f"  Early stopping triggered after {cfg.patience} epochs without improvement.")
        break

print("Training finished.")

# 12. Finalize the wandb run
wandb.finish()

Starting training...
Epoch 1/5


/tmp/ipython-input-646060743.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if cfg.use_amp else None
/tmp/ipython-input-1765662441.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Train Loss: 0.6925
  Val Loss: 0.6239
  Val ROC AUC: 0.7483
  Val Accuracy: 0.6750
  Val Precision: 0.6766
  Val Recall: 0.6750
  Val F1-Score: 0.6743
  Confusion Matrix:
[[314 186]
 [139 361]]
  Saved best model to /content/drive/MyDrive/vit_checkpoints/best_model.pth (Val Loss: 0.6239)
Epoch 2/5


/tmp/ipython-input-1765662441.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Train Loss: 0.5913
  Val Loss: 0.5647
  Val ROC AUC: 0.8421
  Val Accuracy: 0.7570
  Val Precision: 0.7583
  Val Recall: 0.7570
  Val F1-Score: 0.7567
  Confusion Matrix:
[[361 139]
 [104 396]]
  Saved best model to /content/drive/MyDrive/vit_checkpoints/best_model.pth (Val Loss: 0.5647)
Epoch 3/5


...

/tmp/ipython-input-1765662441.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Train Loss: 0.5173
  Val Loss: 0.5251
  Val ROC AUC: 0.8777
  Val Accuracy: 0.7850
  Val Precision: 0.7850
  Val Recall: 0.7850
  Val F1-Score: 0.7850
  Confusion Matrix:
[[390 110]
 [105 395]]
  Saved best model to /content/drive/MyDrive/vit_checkpoints/best_model.pth (Val Loss: 0.5251)
Epoch 5/5


/tmp/ipython-input-1765662441.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Train Loss: 0.5047
  Val Loss: 0.5224
  Val ROC AUC: 0.8799
  Val Accuracy: 0.7920
  Val Precision: 0.7921
  Val Recall: 0.7920
  Val F1-Score: 0.7920
  Confusion Matrix:
[[391 109]
 [ 99 401]]
  Saved best model to /content/drive/MyDrive/vit_checkpoints/best_model.pth (Val Loss: 0.5224)
Training finished.


epoch,▁▃▅▆█
train_loss,█▄▂▁▁
val_accuracy,▁▆▇██
val_f1_score,▁▆▇██
val_loss,█▄▂▁▁
val_precision,▁▆▇██
val_recall,▁▆▇██
val_roc_auc,▁▆▇██
best_val_accuracy,0.792
best_val_loss,0.52239
best_val_roc_auc,0.8799


In [27]:
print("\n--- Evaluating best model on Test Set ---")

# Re-initialize wandb for test evaluation if needed (training run might have ended)
# We'll create a new run specific for logging test results, linking it to the previous run if desired.
# For simplicity, starting a fresh run for test results here.
import wandb
import os

wandb.init(
    project=cfg.project, # Use the same project
    name=f"{cfg.run_name}-test-evaluation", # A new run name for test evaluation
    tags=cfg.tags + ["test-evaluation"],
    config=asdict(cfg) # Log the configuration again
)

# 1. Load the best model
model.load_state_dict(torch.load(os.path.join(cfg.save_dir, 'best_model.pth')))

# 2. Run validation on the test set
test_loss, test_preds, test_labels = validate_one_epoch(
    model, test_dataloader, criterion, device, cfg.use_amp
)

# 3. Calculate metrics for the test set
test_roc_auc = roc_auc_score(test_labels, test_preds)

# Binarize predictions for classification report and confusion matrix
test_bin_preds = (test_preds > 0.5).astype(int)
test_report = classification_report(test_labels, test_bin_preds, output_dict=True, zero_division=0)
test_confusion_matrix = confusion_matrix(test_labels, test_bin_preds)

# Extract key metrics from classification report
test_accuracy = test_report['accuracy']
test_precision = test_report['macro avg']['precision']
test_recall = test_report['macro avg']['recall']
test_f1 = test_report['macro avg']['f1-score']

print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test ROC AUC: {test_roc_auc:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Test Precision: {test_precision:.4f}")
print(f"  Test Recall: {test_recall:.4f}")
print(f"  Test F1-Score: {test_f1:.4f}")
print(f"  Test Confusion Matrix:\n{test_confusion_matrix}")

# 4. Log test metrics to wandb
wandb.log({
    "test_loss": test_loss,
    "test_roc_auc": test_roc_auc,
    "test_accuracy": test_accuracy,
    "test_precision": test_precision,
    "test_recall": test_recall,
    "test_f1_score": test_f1,
    "test_confusion_matrix": wandb.Table(data=test_confusion_matrix.tolist(), columns=["Pred 0", "Pred 1"])
})

print("Test set evaluation complete and metrics logged to Weights & Biases.")

# Finalize the wandb run for test evaluation
wandb.finish()


--- Evaluating best model on Test Set ---


/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Test Loss: 0.5313
  Test ROC AUC: 0.8692
  Test Accuracy: 0.7920
  Test Precision: 0.7923
  Test Recall: 0.7920
  Test F1-Score: 0.7919
  Test Confusion Matrix:
[[388 112]
 [ 96 404]]
Test set evaluation complete and metrics logged to Weights & Biases.


test_accuracy,▁
test_f1_score,▁
test_loss,▁
test_precision,▁
test_recall,▁
test_roc_auc,▁
test_accuracy,0.792
test_f1_score,0.79195
test_loss,0.53133
test_precision,0.7923
test_recall,0.792
